[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# groupby


## What you will be able to do

Split the readings into groups by station, by month or by any key you can compute, read what pandas
holds before you ask for an answer, and get one result for every group in a single expression.

## The idea

### The problem

A large share of the questions asked of a table have the form "for each X, work out Y", such as each
station's mean for the year, each month's coldest hour, or how many readings each station sent. Each
one is the same three steps. Split the rows into parts that share a value of X, work out Y within each part, and
put the answers together, labeled by X.

Written by hand, that is a loop over the stations, a filter inside it, and a dictionary to collect the
answers. The loop makes a full pass over all 35,040 rows for every station, and a question with two
keys, such as each station in each month, needs a loop inside a loop and a key made of a pair.

pandas does the three steps with `groupby`. What surprises people is the first step on its own.
`readings.groupby("station")` prints no numbers at all, only the name of a class and a memory address,
and it is not obvious what that object is or what can be done with it.

### What groupby is

> **`df.groupby(key)`** splits the rows of `df` into groups that share a value of `key`, and returns a
> GroupBy object that holds the key, works out which rows belong to which group the first time it
> is asked, and computes nothing more. A
> method called on that object, such as `.mean()`, runs on every group and **combines** the results
> into one Series or DataFrame indexed by the group keys.

### Why it works that way

Finding which rows belong together is the costly part, so pandas does it once, the first time a summary
or `groups` needs it, and every later summary asked of the same object reuses it. That is why `groupby` returns an object
instead of an answer, and why the same object can give a mean, a count and a maximum in turn.

A column chosen after the split, as in `readings.groupby("station")["celsius"]`, narrows the work to
that column. Without one, a summary runs on every column but the key, and a column the summary cannot
handle raises.

The combined result is indexed by the keys, sorted, so the answer for Svalbard is labeled Svalbard. A
key can be a column's name, a list of names, or a Series computed from the table, such as the month of
every reading. Rows whose key is missing are left out of every group unless you ask otherwise.

`size` counts the rows in each group, and `count`, which looks as if it does the same, counts the values
in a column that are not missing, so Svalbard's 24 silent hours show up as a difference between them.

### Where this shows up

It is SQL's `GROUP BY`, which the **Tables and Queries** notebook in the **sqlite3, Deep Dive** guide
uses. The **Aggregating** notebook asks a GroupBy for several summaries at once, the **Reshaping**
notebook's `pivot_table` is a groupby laid out as a grid, and resampling in the **Dates and Times**
notebook groups readings by time.

### What this notebook covers

- the three steps as a loop, and as one `groupby`
- what a GroupBy object holds, and how to look inside it
- choosing a column, and `size` against `count`
- grouping by two keys and by a key computed from a column
- `as_index=False`, and `apply` for a function of your own
- four errors, one of them silent

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import pandas as pd

readings = pd.DataFrame({
    "station": ["Oslo", "Bergen", "Oslo", "Bergen"],
    "celsius": [6.1, 8.2, 5.8, 7.9],
})
grouped = readings.groupby("station")
print(grouped.ngroups, "groups:", list(grouped.groups))
print(grouped["celsius"].max())
```

```
2 groups: ['Bergen', 'Oslo']
station
Bergen    8.2
Oslo      6.1
Name: celsius, dtype: float64
```

`grouped` found two groups and sorted their keys. `["celsius"].max()` took the largest reading in
each group, and the answer is a Series indexed by station.

## Setup

One library, two helpers for printing, three for building the readings, and two tables.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `re` replaces the memory address in a GroupBy object's printed form, which differs on every run
- `warnings` catches the warning `apply` gives in Common errors, so that `print_warnings` can show its
  category and its first sentence
- `math`, `datetime` and `timedelta` build the readings

`readings` holds every hour of 2025 at the four stations, by the same formula as the **sqlite3, Deep
Dive** guide, with Svalbard's 24 silent hours on 2 March as `NaN`. `stations` holds each station's
latitude, and Kirkenes, which has no readings.

In [1]:
import math
import re
import warnings
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})


def print_warnings(caught):
    """Print each caught warning as its category and the first sentence of its message."""
    for warning in caught:
        first_line = str(warning.message).strip().splitlines()[0]
        print(f"{warning.category.__name__}: {first_line.split('. ')[0].rstrip('.')}.")


print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### Split, apply and combine with a loop

Each station's mean, the long way:

In [2]:
answers = {}
for station in readings["station"].unique():
    rows = readings[readings["station"] == station]          # split
    answers[station] = round(rows["celsius"].mean(), 2)      # apply
pd.Series(answers)                                          # combine

Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
dtype: float64

The loop filtered all 35,040 rows four times, once for each station, and the answers were collected by
hand.

### The same with groupby

In [3]:
readings.groupby("station")["celsius"].mean().round(2)

station
Bergen      8.00
Oslo        6.50
Svalbard   -4.48
Tromso      3.50
Name: celsius, dtype: float64

The same four means, in one expression, with the index named `station` after the key.

### What groupby returns

A GroupBy object printed as it is shows its class and its memory address. The address differs on every
run, so the cell replaces its digits before printing:

In [4]:
grouped = readings.groupby("station")

print(re.sub(r"0x[0-9a-f]+", "0x...", repr(grouped)))
print(grouped.ngroups, "groups:", list(grouped.groups))
print("Oslo's first row labels:", grouped.groups["Oslo"][:3].tolist())

<pandas.core.groupby.generic.DataFrameGroupBy object at 0x...>
4 groups: ['Bergen', 'Oslo', 'Svalbard', 'Tromso']
Oslo's first row labels: [1, 5, 9]


The object holds no means and no counts. It holds the split, which `groups` shows as a dictionary from
each key to the row labels in that group. Oslo's rows are 1, 5, 9 and onward, every fourth row,
because the stations take turns in `readings`.

### Looking inside the groups

Looping over a GroupBy gives each key with its rows as a DataFrame, and `get_group` returns one group:

In [5]:
for station, group in grouped:
    print(f"{station:9} {len(group)} rows, warmest {group['celsius'].max()}")

grouped.get_group("Svalbard").head(3)

Bergen    8760 rows, warmest 20.8
Oslo      8760 rows, warmest 19.3
Svalbard  8760 rows, warmest 8.3
Tromso    8760 rows, warmest 16.2


,station,time,celsius
2,Svalbard,2025-01-01 00:00:00,-14.6
6,Svalbard,2025-01-01 01:00:00,-16.4
10,Svalbard,2025-01-01 02:00:00,-16.4


Each `group` is an ordinary DataFrame, with the original row labels. A loop like this is the way to
look at the groups while you work something out, and the summaries below are the way to get an answer.

### Choosing a column

`grouped["celsius"]` narrows the GroupBy to one column. A summary of the whole GroupBy runs on every
column but the key, here `time` and `celsius`:

In [6]:
print(type(grouped["celsius"]).__name__)

grouped.mean().round(2)

SeriesGroupBy


,time,celsius
station,,
Bergen,2025-07-02 11:30:00,8.00
Oslo,2025-07-02 11:30:00,6.50
Svalbard,2025-07-02 11:30:00,-4.48
Tromso,2025-07-02 11:30:00,3.50


The mean of `time` is 11:30 on 2 July, the middle of the year, which is what an average of every hour of
2025 comes to. That is rarely a useful number, which is one reason to choose the column first.

### size against count

In [7]:
pd.DataFrame({"size": grouped.size(), "count": grouped["celsius"].count()})

,size,count
station,,
Bergen,8760,8760
Oslo,8760,8760
Svalbard,8760,8736
Tromso,8760,8760


`size` counts rows, 8,760 for every station. `count` counts the readings that are not missing, and
Svalbard's is 8,736. A mean divides by the count, so Svalbard's mean is over the hours it reported.

### Grouping by two keys

A list of keys groups by every combination that appears in the rows, and the result has one index level
for each key:

In [8]:
month = readings["time"].dt.month.rename("month")
by_month = readings.groupby(["station", month])["celsius"].mean().round(1)

print(by_month.head(4))
print("Svalbard in July:", by_month.loc[("Svalbard", 7)])

station  month
Bergen   1       -0.9
         2        0.1
         3        3.2
         4        7.5
Name: celsius, dtype: float64
Svalbard in July: 4.4


The second key is a Series, the month of every reading, and it groups by its values just as a column
name would. `loc` with a pair picks one combination, and **The Index** notebook covers indexes of more
than one level.

### A key computed from a column

Grouping by the hour of the day, across all four stations and every day, gives the average shape of a
day:

In [9]:
hour = readings["time"].dt.hour.rename("hour")
by_hour = readings.groupby(hour)["celsius"].mean().round(2)

print("coldest hour of the day:", by_hour.idxmin(), "| warmest:", by_hour.idxmax())
by_hour.head(4)

coldest hour of the day: 3 | warmest: 15


hour
0    1.26
1    0.78
2    0.49
3    0.38
Name: celsius, dtype: float64

03:00 is the coldest hour and 15:00 the warmest, which is where the formula's daily swing has its low
and its high.

### A flat table with as_index=False

`as_index=False` keeps the keys as an ordinary column instead of the index:

In [10]:
readings.groupby("station", as_index=False)["celsius"].max()

,station,celsius
0,Bergen,20.8
1,Oslo,19.3
2,Svalbard,8.3
3,Tromso,16.2


### apply for a function of your own

`apply` calls a function once for every group, with the group's rows as a DataFrame, and combines what
it returns. `include_groups=False` leaves the key column out of the rows it passes in:

In [11]:
def warmest_hour(group):
    """The time of the warmest reading in one group."""
    return group.loc[group["celsius"].idxmax(), "time"]


grouped.apply(warmest_hour, include_groups=False)

station
Bergen     2025-07-21 15:00:00
Oslo       2025-07-17 15:00:00
Svalbard   2025-07-13 15:00:00
Tromso     2025-07-09 15:00:00
dtype: datetime64[ns]

Each station's warmest hour, labeled by station. `apply` is the most flexible summary. It runs a Python
function once for every group, where a built-in method such as `max` runs compiled code, so reach for
it when no built-in method does the job.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/12-groupby-solutions.ipynb).

**1.** Print each station's lowest reading of the year, with `groupby` and `min`.

In [12]:
# your code here

**2.** Count the readings below zero at each station, by filtering first and grouping the result.

In [13]:
# your code here

**3.** Print the mean of each month across all four stations, rounded to one place, by grouping on `readings["time"].dt.month`.

In [14]:
# your code here

**4.** Loop over `readings.groupby("station")` and print each station's name with the number of its readings above 10 degrees.

In [15]:
# your code here

**5.** Make a table, with `as_index=False`, of each station's highest reading, sorted from highest to lowest.

In [16]:
# your code here

## Common errors

### TypeError: datetime64 type does not support sum operations

In [17]:
readings.groupby("station").sum()

TypeError: datetime64 type does not support sum operations

With no column chosen, `sum` runs on every column but the key, and `time` holds dates and times, which
cannot be added, so choose the column first:

In [18]:
readings.groupby("station")["celsius"].sum().round(1)

station
Bergen      70079.0
Oslo        56940.2
Svalbard   -39153.9
Tromso      30659.2
Name: celsius, dtype: float64

### KeyError: 'Kirkenes'

In [19]:
grouped.get_group("Kirkenes")

KeyError: 'Kirkenes'

Kirkenes is in `stations` and has no readings, so `readings.groupby("station")` made no group for it.
A group exists only for a key that appears in the rows. `list(grouped.groups)` shows the keys there
are.

### No error, and a row is left out of every group

Here a reading arrives with no station name:

In [20]:
unlabeled = pd.DataFrame({"station": ["Oslo", None, "Oslo"], "celsius": [6.1, 5.0, 5.8]})

print(len(unlabeled), "rows")
print(unlabeled.groupby("station").size())

3 rows
station
Oslo    2
dtype: int64


Three rows went in, and the groups hold two. A row whose key is missing belongs to no group, and
pandas drops it without saying so. `dropna=False` keeps such rows in a group of their own:

In [21]:
unlabeled.groupby("station", dropna=False).size()

station
Oslo    2
NaN     1
dtype: int64

### DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns.

In [22]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    peaks = grouped.apply(warmest_hour)

print_warnings(caught)
without_groups = grouped.apply(warmest_hour, include_groups=False)
chosen_columns = grouped[["time", "celsius"]].apply(warmest_hour)
print("same as with include_groups=False:", peaks.equals(without_groups))
print("same as choosing the columns:", peaks.equals(chosen_columns))

same as with include_groups=False: True
same as choosing the columns: True


Without `include_groups=False`, `apply` passed each group with its `station` column included, which
pandas will stop doing in a future version, and it warns now. The result is the same here, because
`warmest_hour` never reads `station`, and the last two lines compare it with the two ways to avoid the
warning. Pass `include_groups=False`, as the worked example did, or choose the columns the function
needs after `groupby`, as in `grouped[["time", "celsius"]].apply(warmest_hour)`.

## Recap

- Split a table with `groupby`, and read the GroupBy object as the key and the split, with nothing
  computed until you ask
- Choose a column after the split, and get one labeled result for every group with a method such as
  `mean`, `max` or `count`
- Tell `size`, the rows in a group, from `count`, the values that are not missing
- Group by two keys, or by a Series computed from a column, such as the month or the hour
- Run a function of your own on every group with `apply` and `include_groups=False`

## What is next

The **Aggregating** notebook asks a GroupBy for several summaries at once, names the columns of the
answer, and puts each group's value back beside every row with `transform`.

---

&#8592; **Previous:** [Adding and Changing Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/11-adding-and-changing-columns.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Aggregating](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/13-aggregating.ipynb) &#8594;
